# Experiment: From 3GB to 8MB: What MRL + Binary Quantization actually costs you in retrieval quality

## What This Notebook Investigates
Most teams building semantic search store embeddings at full float32 precision
across hundreds of dimensions. This experiment asks: how much retrieval quality
do you actually lose when you compress aggressively using Matryoshka
Representation Learning (MRL) + Binary Quantization — and what survives?

## Model
**nomic-ai/nomic-embed-text-v1.5**  
Chosen because it is natively trained with MRL — meaning truncating to 64
dimensions is an intended use case. It also has a permissive
open-source license, making every result in this notebook fully reproducible
by anyone.
With 8,192 tokens, Nomic can "read" roughly 12–15 pages of text. For your Amazon dataset, this means it sees the Title, the full Description, and every single technical spec in the Features list. Nothing is ignored.

! Gotcha: This model requires F.layer_norm() before truncation and mandatory
task prefixes (search_document: / search_query:). Both are handled explicitly
in the pipeline. Skipping either silently degrades embedding quality.

## Dataset
**McAuley-Lab/Amazon-Reviews-2023 — Electronics subset (1 shard)**  
Source: Hugging Face Datasets  
Why: Real-world product data with natural semantic clusters
(laptops, cameras, phones, audio). This domain is heavily used in
recommendation systems — exactly where embedding compression decisions
matter most in production.

Note: The full Electronics review dataset is tens of gigabytes as raw .jsonl.
We use only the metadata parquet (one shard = 161,002 rows) and sample 5,000
balanced rows across 4 categories. Balanced to eliminate class imbalance bias. So that we it is being tested on its ability to understand semantics, not just its ability to memorize the most frequent category.

In [ ]:
!pip install sentence-transformers umap-learn matplotlib pyarrow huggingface_hub -q

In [ ]:
# INGESTION PIPELINE: Load ONE shard of Amazon Electronics metadata from HF, clean it, balance it, and save it to Drive.

import pandas as pd
import numpy as np
from huggingface_hub import login
from google.colab import userdata, drive
import os

In [ ]:
login(token=userdata.get('HF_TOKEN'))
print("Authenticated with Hugging Face")

In [ ]:
# 1a. ───────────────────────────────── Load one shard ─────────────────────────────────
# full-00000-of-00010.parquet alone has 161,002 rows, more than enough for this experiment.e
# Why raw_meta_Electronics and not raw_review_Electronics? : Reviews exist only as .jsonl (~8GB) + captures sentiment (might clustr based on that! which we dont want). Meta has parquet + captures ontology, Meta also has richer text (title + description + features) and explicit category labels — better for UMAP visualization.
SHARD_URL = (
    "hf://datasets/McAuley-Lab/Amazon-Reviews-2023"
    "/raw_meta_Electronics/full-00000-of-00010.parquet"
)

# Parquet has 16 columns, we only need 4.
df_raw = pd.read_parquet(
    SHARD_URL,
    engine="pyarrow",
    columns=['title', 'description', 'features', 'main_category']
)

print(f"Loaded shard: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
print(f"  Null main_category: {df_raw['main_category'].isna().sum():,}")

In [ ]:
# 1b. ───────────────────────────────── Convert numpy arrays → clean strings ──────────────────────
# Why is this needed?:  HF stores list-type fields (description, features) as numpy arrays inside parquet cells.
# Example:
#   array(['Great product for FPV...'], dtype=object)
#   array([], dtype=object)   ← empty, not NaN

def array_to_text(val):
    if isinstance(val, np.ndarray):
        return ' '.join(
            str(s).strip()
            for s in val
            if pd.notna(s) and str(s).strip()
        )

    # Handle scalar missing values
    if pd.isna(val):   # None, NaN, NaT
        return ''

    return str(val).strip()

In [ ]:
df_raw['title_clean']    = df_raw['title'].apply(array_to_text) # Wont scale apply (under the hood for loop; vectorized string operation is not straigtforward here)
df_raw['desc_clean']     = df_raw['description'].apply(array_to_text)
df_raw['features_clean'] = df_raw['features'].apply(array_to_text)

print("   Arrays converted to strings")
print(f"  Sample title:    {df_raw['title_clean'].iloc[0][:80]}")
print(f"  Sample desc:     {df_raw['desc_clean'].iloc[0][:80]}")
print(f"  Sample features: {df_raw['features_clean'].iloc[0][:80]}")

In [ ]:
# 1c. ───────────────────────────────── Build unified text column ─────────────────────────────────
# Why combine title + description + features?
# Each field carries different signal:
#   title    → product name, model number, key specs (always present)
#   desc     → marketing copy, use cases, context (often rich)
#   features → bullet points, specs, compatibility (structured info)
# Nomic's have Max Sequence Length (8,192)!!

# The embedding model sees all three together — giving it the
# fullest possible picture of what the product is.
# The ' | ' separator is a clean visual break; the model treats
# it as a soft boundary, not a hard signal. (special token like [SEP])
#
# Why len > 0 check before joining?
# Avoids joining empty strings as ' | ' artifacts like "title | | "

def build_text(row):
    parts = [
        row['title_clean'],
        row['desc_clean'],
        row['features_clean']
    ]
    return ' | '.join(p for p in parts if len(p) > 0)

df_raw['text'] = df_raw.apply(build_text, axis=1) # new Pandas series object for every single row (need better approach if scaling) + axis=1 will use slowest ops in Pandas
df_raw['text_len'] = df_raw['text'].str.len()

print("   Text column built")
print(f"  Mean length:  {df_raw['text_len'].mean():.0f} chars")
print(f"  Median length:{df_raw['text_len'].median():.0f} chars")
print(f"  Empty texts:  {(df_raw['text_len'] < 20).sum():,}")

In [ ]:
# ── 1e. Map messy categories → clean labels ───────────────────────
# Why hand-map instead of using raw values?
# main_category has inconsistent naming:
#   'AMAZON FASHION', 'Amazon Home' → wrong domain entirely
#   'All Electronics' → too broad, no semantic meaning
#   'Computers' → same as 'Computers & Laptops' but shorter
#
# We map to None what we want to DROP.
# We rename to consistent human-readable labels what we KEEP.


CATEGORY_MAP = {
    'Computers':                     'Computers & Laptops',
    'Cell Phones & Accessories':     'Mobile & Phones',
    'Camera & Photo':                'Camera & Photo',
    'Home Audio & Theater':          'Audio & Home Theater',
    'Portable Audio & Accessories':  'Audio & Home Theater',  # merged
    'All Electronics':               None,   # too broad
    'Industrial & Scientific':       None,   # off-topic
    'AMAZON FASHION':                None,   # wrong domain
    'Amazon Home':                   None,   # too broad
    'Office Products':               None,   # too broad
    'Musical Instruments':           None,   # too small
    'Sports & Outdoors':             None,   # off-topic
    'GPS & Navigation':              None,   # only 782 rows → thin cluster
}

df_raw['category'] = df_raw['main_category'].map(CATEGORY_MAP)

print(" Categories mapped")
print(df_raw['category'].value_counts(dropna=False))

In [ ]:
# ── 1f. Filter and balance ────────────────────────────────────────
# Why text_len > 80?
# Texts under 80 chars are usually just a product title with
# no description or features. They embed poorly — the model
# has almost no context to work with. These create noise in UMAP.
#
# Why 5,000 per category?
# Enough for UMAP to form dense, convincing clusters.
# 4 categories × 5,000 = 20,000 rows total.
# safe on free T4 (~10 min, ~6GB RAM)


KEEP_CATEGORIES = [
    'Computers & Laptops',   # 46,566 available → take 5,000
    'Camera & Photo',        # 21,511 available → take 5,000
    'Mobile & Phones',       # 15,583 available → take 5,000
    'Audio & Home Theater',  # 11,335 available → take 5,000
]

N_PER_CATEGORY = 5_000

df_filtered = (
    df_raw[df_raw['category'].isin(KEEP_CATEGORIES)]
    .query("text_len > 80") # numexpr under the hood
    .dropna(subset=['category', 'text'])
)

# Verify we have enough before sampling
print("=== Available after filter (must be > 5000 each) ===")
print(df_filtered['category'].value_counts())


# Sample only if all categories have enough rows
counts = df_filtered['category'].value_counts()
for cat in KEEP_CATEGORIES:
    available = counts.get(cat, 0)
    status = "✓" if available >= N_PER_CATEGORY else "✗ NOT ENOUGH"
    print(f"{status}  {cat}: {available:,} available")

df_sampled = (
    df_filtered
    .groupby('category')
    .apply(lambda x: x.sample(n=N_PER_CATEGORY, random_state=42))
    .reset_index(drop=True) # shreds messy multi-level index
    [['text', 'category']]
)

print(f"\n✓ Final dataset: {df_sampled.shape}")
print(df_sampled['category'].value_counts())

In [ ]:
# ── 1g. Save to Drive ─────────────────────────────────────────────
# This was a network-heavy operation (HF download) -> save to drive
#
# Why two files?
# electronics_meta_200_test.parquet → for testing all downstream cells on CPU without loading 8K rows. Fast iteration.
# electronics_meta_clean.parquet   → the real dataset for GPU run.

drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/experiments/embeddings/01_mrl_binary_quantization/"
os.makedirs(SAVE_PATH, exist_ok=True)

df_sampled.to_parquet(
    f"{SAVE_PATH}electronics_meta_clean.parquet",
    index=False
)
df_sampled.head(200).to_parquet(
    f"{SAVE_PATH}electronics_meta_200_test.parquet",
    index=False
)

print(f"✓ Saved full dataset  → electronics_meta_clean.parquet")
print(f"✓ Saved test dataset  → electronics_meta_200_test.parquet")
print(f"\nDone. We would not need to touch HF or run this step again.")
print(f"All future sessions load from Drive.")

In [ ]:
# Step 2: Embedding Pipeline
# What this step does and why each decision exists.
# We take the clean 20K product texts, pass them through Nomic v1.5, and save the raw float32 embeddings to Drive. That's it.
# This is the only GPU step in the entire experiment. Everything after this — compression, UMAP, retrieval scoring — runs on CPU from saved files.

# Three things that make Nomic v1.5 different from any other model:
# Task prefix is mandatory. search_document: changes the embedding meaningfully — not a suggestion, it's part of the model's training contract.
# Without it your embeddings are worse and you're not using the model as designed.
# Layer norm before truncation. Unique to Nomic. Other MRL models let you slice directly. Nomic requires F.layer_norm first or the truncated dimensions are miscalibrated.
# Normalize after truncation. After slicing to 64 dims, renormalize to unit length. Cosine similarity breaks on unnormalized vectors.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2: EMBEDDING PIPELINE

# ── SMOKE TEST: 50 rows, CPU, no GPU needed ───────────────────────
# Goal: verify the entire embedding pipeline works end to end
# before touching GPU quota.
# If anything breaks (import error, shape error, norm error)
# we catch it here.

# ── 2a. Installs ──────────────────────────────────────────────────
# Why einops?
# Nomic v1.5 uses it internally for tensor operations.
# Without it the model throws an ImportError on load.
!pip install sentence-transformers einops -q

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from google.colab import drive
import time

# Confirm we're on CPU — expected here
print(f"CUDA available: {torch.cuda.is_available()}")  # should print False
print(f"Device: CPU")

In [ ]:
# Load from Drive
df = pd.read_parquet(f"{SAVE_PATH}electronics_meta_200_test.parquet")

# Take only 50 rows for smoke test
df_test = df.head(50).reset_index(drop=True)
print(f"✓ Loaded {len(df_test)} rows for smoke test")
print(df_test['category'].value_counts())

In [ ]:
# Load model on CPU
print("Loading Nomic v1.5 on CPU...")
print("(slower than GPU but works fine for 50 rows)")

model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True,
    device='cpu'          # explicit CPU
)
print(f"✓ Model loaded")

In [ ]:
# Add prefix
texts = [f"search_document: {text}" for text in df_test['text'].tolist()]
print(f"✓ {len(texts)} texts prefixed")
print(f"  Sample: {texts[0][:100]}...")

In [ ]:
# Encode — small batch size for CPU
# batch_size=8 on CPU is safe and avoids memory pressure
# 50 rows ÷ 8 = 7 batches — will finish in 1-3 minutes on CPU

print("Encoding 50 rows on CPU...")
t0 = time.time()

# ~4KB per vector, for 20K => 60 MB
embeddings_raw = model.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=False,
    convert_to_numpy=True
)

elapsed = time.time() - t0
print(f"\n✓ Done in {elapsed:.1f} seconds")
print(f"  Shape: {embeddings_raw.shape}")    # expect (50, 768)
print(f"  Dtype: {embeddings_raw.dtype}")    # expect float32

In [ ]:
# Layer norm → L2 normalize
embeddings_tensor = torch.tensor(embeddings_raw)

embeddings_normed = F.layer_norm(
    embeddings_tensor,
    normalized_shape=(embeddings_tensor.shape[1],)
).numpy()

embeddings_final = embeddings_normed / np.linalg.norm(
    embeddings_normed, axis=1, keepdims=True
)

print(f"✓ Layer norm + L2 done")
print(f"  Shape:  {embeddings_final.shape}")
print(f"  Norms (all should be 1.0): {np.linalg.norm(embeddings_final[:5], axis=1).round(4)}")

In [ ]:
# Save smoke test embeddings — separate file, don't overwrite real data
np.save(f"{SAVE_PATH}smoke_test_embeddings.npy", embeddings_final)
np.save(f"{SAVE_PATH}smoke_test_labels.npy", df_test['category'].values)

print(f"✓ Saved smoke test embeddings")
print(f"\n{'='*50}")
print(f"SMOKE TEST COMPLETE")
print(f"  Input:  {len(texts)} texts")
print(f"  Output: {embeddings_final.shape} float32")
print(f"  Norms:  all 1.0 ✓")
print(f"{'='*50}")
print(f"\nIf everything above looks correct →")
print(f"Switch to T4 GPU runtime and run the full 20K pipeline.")

In [ ]:
# Goal: Encode 20K product texts → float32 embeddings → save.
#
# GPU needed: YES — this is the ONLY GPU cell in the experiment.
# Expected time: ~10 minutes on T4 for 20K rows.
# Expected RAM: ~6GB GPU VRAM, ~4GB system RAM.
#
# After this cell completes:
#   - Switch back to CPU runtime
#   - All remaining steps load from Drive, no GPU needed

In [ ]:
# ── Verify GPU before touching anything else ──────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── OOM Crash Fix1: Clear everything before starting fresh ─────────────────
import torch
import gc

# Kill whatever is sitting in VRAM from the failed run
torch.cuda.empty_cache()
gc.collect()

print(f"VRAM reserved after clear: {torch.cuda.memory_reserved(0)/1e9:.2f} GB reserved")
print(f"VRAM allocated after clear: {torch.cuda.memory_allocated(0)/1e9:.2f} GB allocated")

In [ ]:
# ── 2c. Load clean dataset from Drive ─────────────────────────────
# Why from Drive and not re-downloading?
# Step 1 saved the clean 20K sample. We never touch HF again.
# This loads in under 1 second vs minutes of network IO.

!pip install sentence-transformers einops -q

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from google.colab import drive
import time
import os

drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/experiments/embeddings/01_mrl_binary_quantization/"

df = pd.read_parquet(f"{SAVE_PATH}electronics_meta_clean.parquet")
print(f"✓ Loaded: {df.shape}")
print(df['category'].value_counts())

In [ ]:
# ── OOM Fix 1: Set memory allocator config BEFORE loading model ───────
# expandable_segments=True prevents fragmentation; instead of grabbing one giant contiguous block,
# PyTorch uses smaller expandable segments. -> Prevents free but not free OOM issue

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("✓ Memory allocator configured")

In [ ]:
# ── 2d. Load Nomic v1.5 ───────────────────────────────────────────
# Nomic v1.5 uses a custom model class not in the standard
# sentence-transformers library. This flag allows loading it.
# This is safe here — Nomic is a reputable open-source model.
#
# Explicitly forces GPU. Without this, sentence-transformers
# sometimes falls back to CPU silently on Colab.

print("Loading Nomic v1.5...")
model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True,
    device='cuda'
)
print(f"✓ Model loaded on GPU")
print(f"  Max sequence length: {model.max_seq_length}")
print(f"  Output dimension:    768")

In [ ]:
# ── 2e. Add task prefix ───────────────────────────────────────────
# Nomic v1.5 was trained with task-specific prefixes.
# The model routes text into different subspaces depending
# on the prefix. For a corpus of documents you're indexing,
# 'search_document:' is correct.
# 'search_query:' is for the query side — used at retrieval time.
#
# This is not cosmetic. In benchmarks, wrong or missing prefix
# drops retrieval quality measurably.

texts = [
    f"search_document: {text}"
    for text in df['text'].tolist()
]

print(f"✓ Prefix added to {len(texts):,} texts")
print(f"  Sample: {texts[0][:120]}...")

In [ ]:
# ── OOM Fix2: Reduce batch size from 64 → 16 ────────────────────────
# As seen above texts are long — mean 1,134 chars.
# batch_size=64 × long texts = huge intermediate tensors.
# batch_size=16 uses ~4x less VRAM per batch -> In Transformer models, memory usage grows quadratically with sequence length O(n^2)
# Slower but completes without OOM.
#
# OOM Fix3: encoded in two halves as a safety net.
# If it OOMs at batch 200 of 312, we lose everything.
# Two halves means worst case we only redo half.

import numpy as np
import torch.nn.functional as F
import time

texts = [f"search_document: {text}" for text in df['text'].tolist()]

half = len(texts) // 2
texts_a = texts[:half]
texts_b = texts[half:]

print(f"Half A: {len(texts_a):,} texts")
print(f"Half B: {len(texts_b):,} texts")
print(f"Encoding with batch_size=16...")

In [ ]:
# ── 2f. Encode in batches ─────────────────────────────────────────
# 10K rows at batch 16 per half = 625 batches.
# We do NOT normalize here. We need the raw float32 values
# for the layer_norm step that comes next.
# Normalizing before layer_norm would corrupt the MRL truncation.

print("Encoding... (this takes ~8-10 mins on T4)")
t0 = time.time()

emb_a = model.encode(
    texts_a,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=False,   # raw values — layer_norm comes next
    convert_to_numpy=True
)

print(f"✓ Half A done: {emb_a.shape} in {(time.time()-t0)/60:.1f} min")
# Clear cache between halves
torch.cuda.empty_cache()
gc.collect()


In [ ]:
# Encode half B
t0 = time.time()
emb_b = model.encode(
    texts_b,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=False,
    convert_to_numpy=True
)
print(f"✓ Half B done: {emb_b.shape} in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── 2g. Layer norm — Nomic-specific requirement ───────────────────
# Why layer_norm before truncation?
# This is the single most important Nomic-specific gotcha.
#
# Standard MRL models distribute information such that slicing
# the first N dims works directly. Nomic v1.5 does NOT.
# Its raw output dimensions are not calibrated for direct slicing.
# Layer_norm rescales each dimension relative to the full vector's
# mean and variance; this recalibration is what makes the first
# 64 dims carry meaningful compressed information.
#
# Without this step: truncated embeddings are garbage.
# With this step: 64-dim embeddings retain ~97% retrieval quality.

embeddings_tensor = torch.tensor(embeddings_raw)

embeddings_normed = F.layer_norm(
    embeddings_tensor,
    normalized_shape=(embeddings_tensor.shape[1],)
).numpy()

embeddings_final = embeddings_normed / np.linalg.norm(
    embeddings_normed, axis=1, keepdims=True
)

print(f"✓ Layer norm + L2 done")
print(f"  Shape:  {embeddings_final.shape}")
print(f"  Norms:  {np.linalg.norm(embeddings_final[:5], axis=1).round(4)}")

In [ ]:
# ── 2h. L2 normalize the full 768-dim embedding ───────────────────
# Layer_norm does NOT produce unit vectors.
# For cosine similarity to work correctly the vectors must
# have norm = 1.0. L2 normalization enforces this.
#
# We save this as our baseline (768-dim, float32, normalized).
# All compression versions are derived from this saved file.

In [ ]:
# ── 2i. Save everything to Drive immediately ──────────────────────
# embeddings.npy → pure float32 matrix, shape (20000, 768)
# labels.npy     → category string per row, shape (20000,)
#
# Keeping them separate means we can load either independently.

# .npy is the native numpy format — fastest possible load,
# no parsing, no type conversion. For a float32 matrix this
# is the right choice. Load time: under 1 second.

# GPU job is done after this
np.save(f"{SAVE_PATH}embeddings_768_float32.npy", embeddings_final)
np.save(f"{SAVE_PATH}labels.npy", df['category'].values)

for fname in ['embeddings_768_float32.npy', 'labels.npy']:
    size_mb = os.path.getsize(f"{SAVE_PATH}{fname}") / 1e6
    print(f"✓ {fname}: {size_mb:.1f} MB")

print("\n✓ GPU work is done. Switch back to CPU runtime now.")
print("  All remaining steps load from Drive — no GPU needed.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3: COMPRESSION PIPELINE
#
# Goal: Derive all 5 compression versions from saved embeddings.
# GPU needed: NO — pure numpy operations
# Expected time: under 30 seconds
# Expected RAM: ~500MB peak
# ═══════════════════════════════════════════════════════════════

import numpy as np
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/experiments/embeddings/01_mrl_binary_quantization/"



In [ ]:
# ── 3a. Load and verify baseline embeddings ───────────────────────
# If the file got corrupted or partially saved, every downstream
# step produces wrong results silently. 30 seconds of checks
# here saves hours of confusion later.

embeddings = np.load(
    f"{SAVE_PATH}embeddings_768_float32.npy",
    allow_pickle=False    # float32 matrix — no pickle needed
)
labels = np.load(
    f"{SAVE_PATH}labels.npy",
    allow_pickle=True     # string array — needs pickle
)

# Validate shape
assert embeddings.shape == (20000, 768), \
    f"Wrong shape: {embeddings.shape}"

# Validate norms — all should be 1.0 from Step 2
norms = np.linalg.norm(embeddings, axis=1)
assert np.allclose(norms, 1.0, atol=1e-5), \
    f"Embeddings not normalized. Min norm: {norms.min():.4f}"

# Validate labels align with embeddings
assert len(labels) == len(embeddings), \
    f"Label count {len(labels)} != embedding count {len(embeddings)}"

print(f"✓ Baseline loaded and verified")
print(f"  Shape:  {embeddings.shape}")
print(f"  Dtype:  {embeddings.dtype}")
print(f"  Norms:  min={norms.min():.4f} max={norms.max():.4f}")
print(f"  Labels: {np.unique(labels)}")

In [ ]:
# ── 3b. Create float32 truncations ───────────────────────────────
# Because Nomic v1.5 is natively MRL-trained AND we applied
# layer_norm in Step 2. These two things together ensure the
# first N dimensions carry the most important information.
# On a non-MRL model, slicing would produce garbage.
#
# Slicing changes the vector length (norm). A 768-dim unit vector
# sliced to 64 dims is no longer unit length — the remaining
# dims don't sum to 1.0 anymore. Cosine similarity requires
# unit vectors. Without renormalization your similarity scores
# are wrong.


def truncate_and_normalize(emb, dim):
    """Slice first `dim` dimensions and renormalize to unit length."""
    sliced = emb[:, :dim]                                    # slice
    norms  = np.linalg.norm(sliced, axis=1, keepdims=True)  # compute new norms
    return sliced / norms                                    # renormalize

emb_256 = truncate_and_normalize(embeddings, 256)
emb_128 = truncate_and_normalize(embeddings, 128)
emb_64  = truncate_and_normalize(embeddings, 64)

# Verify norms after truncation
for name, emb in [('256-dim', emb_256), ('128-dim', emb_128), ('64-dim', emb_64)]:
    norms = np.linalg.norm(emb, axis=1)
    print(f"✓ {name}: shape={emb.shape} | "
          f"norms min={norms.min():.4f} max={norms.max():.4f}")

In [ ]:
# ── 3c. Create binary quantization from 64-dim ────────────────────
# Why binarize from 64-dim and not from 768-dim?
#    This is the MRL + Binary combination we're testing.
#    MRL gives us a compact meaningful 64-dim vector.
#    Binary then compresses the storage further.
#    They work together — MRL first, binary second.
#
# How binarization works:
# Each float32 value → 1 if positive (> 0), → 0 if negative (≤ 0)
# We use the sign of the value after layer_norm.
# This works because layer_norm centers values around 0,
# so the sign carries real directional information.
#
# Storage: 64 bits = 8 bytes per vector (using uint8, 8 bits each)
# Comparison: Hamming distance via XOR + popcount


emb_64_binary = (emb_64 > 0).astype(np.uint8)

# Verify
print(f"✓ Binary: shape={emb_64_binary.shape} | dtype={emb_64_binary.dtype}")
print(f"  Unique values: {np.unique(emb_64_binary)}")  # must be only [0, 1]
print(f"  Mean activation (% of 1s): "
      f"{emb_64_binary.mean()*100:.1f}%")
# Healthy range: 40-60%. Far from 50% means poor binarization.

In [ ]:
# ── 3d. Compute and display compression table ─────────────────────
# We compute actual byte sizes
#
# Why itemsize?
# numpy's .itemsize gives bytes per element for that dtype.
# float32 → 4 bytes, uint8 → 1 byte.
# Total bytes = rows × dims × itemsize.

def compression_stats(name, emb, baseline_bytes):
    rows, dims   = emb.shape
    bytes_vec    = dims * emb.itemsize          # bytes per vector
    total_mb     = (rows * bytes_vec) / 1e6     # total for 20K
    per_1m_gb    = (1_000_000 * bytes_vec) / 1e9  # extrapolated to 1M
    ratio        = baseline_bytes / bytes_vec   # compression vs baseline

    return {
        'Version'       : name,
        'Dims'          : dims,
        'Dtype'         : str(emb.dtype),
        'Bytes/vector'  : bytes_vec,
        '20K total (MB)': round(total_mb, 1),
        '1M vectors(GB)': round(per_1m_gb, 2),
        'Compression'   : f"{ratio:.0f}×"
    }

import pandas as pd

baseline_bytes = 768 * 4  # 768 dims × 4 bytes float32 = 3,072

stats = [
    compression_stats("768-dim float32 (baseline)", embeddings,  baseline_bytes),
    compression_stats("256-dim float32",             emb_256,     baseline_bytes),
    compression_stats("128-dim float32",             emb_128,     baseline_bytes),
    compression_stats("64-dim  float32",             emb_64,      baseline_bytes),
    compression_stats("64-dim  binary",              emb_64_binary, baseline_bytes),
]

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

In [ ]:
# ── 3e. Save all versions ─────────────────────────────────────────
# Keeping them separate means each step loads only what it needs.

versions = {
    'embeddings_256_float32' : emb_256,
    'embeddings_128_float32' : emb_128,
    'embeddings_064_float32' : emb_64,
    'embeddings_064_binary'  : emb_64_binary,
}

for fname, emb in versions.items():
    path     = f"{SAVE_PATH}{fname}.npy"
    np.save(path, emb)
    size_mb  = os.path.getsize(path) / 1e6
    print(f"✓ {fname}.npy: {size_mb:.1f} MB")

print(f"\nAll versions saved. Step 3 complete.")

In [ ]:
# ── 3f. True 1-bit packing with np.packbits ───────────────────────
# uint8 stores each binary value (0 or 1) in a full byte — 8 bits.
# 7 of those 8 bits are wasted (always 0).
# np.packbits packs 8 binary values into a single uint8 byte,
# using all 8 bits. No waste.
#
# Example for one vector of 64 dims:
#   uint8:     64 values × 1 byte = 64 bytes
#   packbits:  64 values ÷ 8 bits = 8 bytes  (8× smaller than uint8)


emb_64_packed = np.packbits(emb_64_binary, axis=1)

print(f"✓ Packed binary: shape={emb_64_packed.shape}")
# shape should be (20000, 8) — 64 bits ÷ 8 = 8 bytes per vector

print(f"  dtype: {emb_64_packed.dtype}")        # uint8
print(f"  bytes/vector: {emb_64_packed.shape[1] * emb_64_packed.itemsize}")  # 8

In [ ]:
# ── 3g. Unpack for verification — can we recover the original? ────
# Packing is lossy in ordering if done wrong. Verifying that
# unpacking gives back identical binary values before trusting
# this for retrieval scoring.

emb_64_unpacked = np.unpackbits(emb_64_packed, axis=1)

# Are they identical to what we packed?
assert np.array_equal(emb_64_unpacked, emb_64_binary), \
    "Round-trip failed — packed != unpacked"

print(f"✓ Round-trip verified: pack → unpack → identical to original")
print(f"  Original binary shape: {emb_64_binary.shape}")
print(f"  Unpacked shape:        {emb_64_unpacked.shape}")

In [ ]:
# ── 3h. Updated compression table with packed binary ──────────────

def compression_stats_v2(name, shape, dtype, bytes_per_vec, baseline_bytes):
    rows         = shape[0]
    total_mb     = (rows * bytes_per_vec) / 1e6
    per_1m_gb    = (1_000_000 * bytes_per_vec) / 1e9
    ratio        = baseline_bytes / bytes_per_vec
    return {
        'Version'        : name,
        'Shape'          : str(shape),
        'Dtype'          : str(dtype),
        'Bytes/vector'   : bytes_per_vec,
        '20K total (MB)' : round(total_mb, 2),
        '1M vectors (GB)': round(per_1m_gb, 3),
        'Compression'    : f"{ratio:.0f}×"
    }

baseline_bytes = 768 * 4  # 3,072 bytes

stats = [
    compression_stats_v2(
        "768-dim float32 (baseline)",
        embeddings.shape, embeddings.dtype,
        768 * 4, baseline_bytes),
    compression_stats_v2(
        "256-dim float32",
        emb_256.shape, emb_256.dtype,
        256 * 4, baseline_bytes),
    compression_stats_v2(
        "128-dim float32",
        emb_128.shape, emb_128.dtype,
        128 * 4, baseline_bytes),
    compression_stats_v2(
        "64-dim float32",
        emb_64.shape, emb_64.dtype,
        64 * 4, baseline_bytes),
    compression_stats_v2(
        "64-dim uint8 binary",
        emb_64_binary.shape, emb_64_binary.dtype,
        64 * 1, baseline_bytes),           # 64 bytes — what we had
    compression_stats_v2(
        "64-dim packed binary (true 1-bit)",
        emb_64_packed.shape, emb_64_packed.dtype,
        8 * 1, baseline_bytes),            # 8 bytes — true 1-bit
]

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))

In [ ]:
# ── 3i. Save packed binary ────────────────────────────────────────
# Save both uint8 and packed versions.
# uint8 → used in retrieval scoring (easier to work with)
# packed → used in the compression table
# retreival scoring  uses uint8 for Hamming distance — unpacking mid-retrieval
# adds complexity with no benefit for a correctness experiment.


np.save(f"{SAVE_PATH}embeddings_064_packed_binary.npy", emb_64_packed)
size_mb = os.path.getsize(f"{SAVE_PATH}embeddings_064_packed_binary.npy") / 1e6
print(f"✓ embeddings_064_packed_binary.npy: {size_mb:.2f} MB")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# RETRIEVAL QUALITY SCORING
#
# Goal: Category Precision: for each compression version, we measure how many of the
#       baseline's top-10 neighbors it correctly recovers.
#
# GPU needed: NO
# Expected time: 2-4 minutes on CPU
# Expected RAM: ~1GB peak
# ═══════════════════════════════════════════════════════════════


In [ ]:
# ── 4a. Load all versions ─────────────────────────────────────────

embeddings     = np.load(f"{SAVE_PATH}embeddings_768_float32.npy", allow_pickle=False)
emb_256        = np.load(f"{SAVE_PATH}embeddings_256_float32.npy", allow_pickle=False)
emb_128        = np.load(f"{SAVE_PATH}embeddings_128_float32.npy", allow_pickle=False)
emb_64         = np.load(f"{SAVE_PATH}embeddings_064_float32.npy", allow_pickle=False)
emb_64_binary  = np.load(f"{SAVE_PATH}embeddings_064_binary.npy",  allow_pickle=False)
labels         = np.load(f"{SAVE_PATH}labels.npy", allow_pickle=True)

print("✓ All versions loaded")
print(f"  Baseline:     {embeddings.shape}")
print(f"  256-dim:      {emb_256.shape}")
print(f"  128-dim:      {emb_128.shape}")
print(f"  64-dim:       {emb_64.shape}")
print(f"  64-dim binary:{emb_64_binary.shape}")

In [ ]:
# ── 4b. Core retrieval function ───────────────────────────────────
# All our vectors are L2-normalized (norm = 1.0).
# For unit vectors: cosine_similarity(a, b) = dot_product(a, b)
# So similarity matrix = embeddings @ embeddings.T
#
# Why exclude self (i != query_idx)?
# Every vector is most similar to itself (score = 1.0).
# If we don't exclude it, every query's top-1 is always itself
# and the scores are artificially inflated.

def get_top_k_neighbors(query_emb, corpus_emb, query_indices, k=10):
    """
    For each query index, find top-k neighbors in corpus.
    Returns dict: {query_idx: [neighbor_idx_1, ..., neighbor_idx_k]}
    Excludes the query itself from results.
    """
    results = {}

    for idx in query_indices:
        query_vec = query_emb[idx]                        # shape (dims,)

        # Dot product with all corpus vectors simultaneously
        scores = corpus_emb @ query_vec                   # shape (n_corpus,)

        # Exclude self — set own score to -inf so it never appears
        scores[idx] = -np.inf

        # Get indices of top-k highest scores
        top_k = np.argpartition(scores, -k)[-k:]          # fast top-k
        top_k = top_k[np.argsort(scores[top_k])[::-1]]   # sort by score

        results[idx] = top_k.tolist()

    return results

In [ ]:
# ── 4c. Hamming distance for binary retrieval ─────────────────────
# Binary vectors are uint8 (0s and 1s).
# Cosine similarity on binary vectors is meaningless —
# the dot product just counts matching 1s, ignoring 0-0 matches.
#
# Hamming distance is the correct metric:
# Count positions where two binary vectors DIFFER.
# Lower hamming = more similar (opposite of cosine).
#
# How it works:
# XOR two binary vectors → 1 where they differ, 0 where same
# Sum the 1s → that's the hamming distance
#
# Packed binary needs hardware popcount to be fast.
# For a correctness experiment on 20K vectors, unpacked uint8
# XOR is fast enough on CPU and much simpler to implement.

def get_top_k_neighbors_hamming(query_emb, corpus_emb, query_indices, k=10):
    results = {}

    for idx in query_indices:
        query_vec = query_emb[idx]

        # XOR + sum = hamming distance
        distances = np.sum(corpus_emb ^ query_vec, axis=1)

        # Fix: uint8 can't hold np.inf
        # Max possible hamming distance = number of dims = 64
        # Setting to 64 ensures self is never in top-k
        distances[idx] = 64

        top_k = np.argpartition(distances, k)[:k]
        top_k = top_k[np.argsort(distances[top_k])]

        results[idx] = top_k.tolist()

    return results

In [ ]:
# ── 4d. Recall@10 scoring function ───────────────────────────────
# What is Recall@10?
# For each query: what fraction of the baseline's top-10 neighbors
# appear in the compressed version's top-10?
#
# Example:
#   Baseline top-10:  [4, 7, 12, 33, 41, 55, 67, 88, 91, 102]
#   Compressed top-10:[4, 7, 12, 33, 41, 55, 67, 88, 95, 110]
#   Overlap: 8 out of 10 → Recall@10 = 0.80 for this query
#
# Final score = mean Recall@10 across all 200 queries.

def recall_at_k(baseline_results, compressed_results, k=10):
    """
    Compute mean Recall@k across all queries.
    baseline_results and compressed_results are dicts:
    {query_idx: [neighbor_indices]}
    """
    scores = []

    for idx in baseline_results:
        baseline_set    = set(baseline_results[idx][:k])
        compressed_set  = set(compressed_results[idx][:k])
        overlap         = len(baseline_set & compressed_set)
        scores.append(overlap / k)

    return np.mean(scores)

In [ ]:
# ── 4e. Run the evaluation ────────────────────────────────────────
# Using 200 queries:
# Enough for statistical stability and variance across random seeds
# is under 1% at this sample size.
# Fast enough on CPU — under 3 minutes for all 5 versions.


import time

N_QUERIES = 200
K = 10

np.random.seed(42)
query_indices = np.random.choice(len(embeddings), N_QUERIES, replace=False)

print(f"Running retrieval evaluation")
print(f"  Queries:  {N_QUERIES}")
print(f"  Top-K:    {K}")
print(f"  Corpus:   {len(embeddings):,} vectors")
print(f"  Versions: 5\n")

# Baseline (ground truth)
t0 = time.time()
baseline_results = get_top_k_neighbors(
    embeddings, embeddings, query_indices, k=K
)
print(f"✓ Baseline (768-dim):  {time.time()-t0:.1f}s")

# 256-dim
t0 = time.time()
results_256 = get_top_k_neighbors(emb_256, emb_256, query_indices, k=K)
recall_256  = recall_at_k(baseline_results, results_256, k=K)
print(f"✓ 256-dim float32:     {time.time()-t0:.1f}s  |  Recall@10 = {recall_256:.4f}")

# 128-dim
t0 = time.time()
results_128 = get_top_k_neighbors(emb_128, emb_128, query_indices, k=K)
recall_128  = recall_at_k(baseline_results, results_128, k=K)
print(f"✓ 128-dim float32:     {time.time()-t0:.1f}s  |  Recall@10 = {recall_128:.4f}")

# 64-dim float32
t0 = time.time()
results_64  = get_top_k_neighbors(emb_64, emb_64, query_indices, k=K)
recall_64   = recall_at_k(baseline_results, results_64, k=K)
print(f"✓  64-dim float32:     {time.time()-t0:.1f}s  |  Recall@10 = {recall_64:.4f}")

# 64-dim binary (Hamming)
t0 = time.time()
results_bin = get_top_k_neighbors_hamming(
    emb_64_binary, emb_64_binary, query_indices, k=K
)
recall_bin  = recall_at_k(baseline_results, results_bin, k=K)
print(f"✓  64-dim binary:      {time.time()-t0:.1f}s  |  Recall@10 = {recall_bin:.4f}")

In [ ]:
# ── 4f. Final results table ───────────────────────────────────────
# This table combined with the compression table

results_data = [
    {
        'Version'          : '768-dim float32 (baseline)',
        'Bytes/vector'     : 3072,
        '1M vectors (GB)'  : 3.072,
        'Compression'      : '1×',
        'Recall@10'        : 1.0000,
        'Quality retained' : '100%'
    },
    {
        'Version'          : '256-dim float32',
        'Bytes/vector'     : 1024,
        '1M vectors (GB)'  : 1.024,
        'Compression'      : '3×',
        'Recall@10'        : round(recall_256, 4),
        'Quality retained' : f"{recall_256*100:.1f}%"
    },
    {
        'Version'          : '128-dim float32',
        'Bytes/vector'     : 512,
        '1M vectors (GB)'  : 0.512,
        'Compression'      : '6×',
        'Recall@10'        : round(recall_128, 4),
        'Quality retained' : f"{recall_128*100:.1f}%"
    },
    {
        'Version'          : '64-dim float32',
        'Bytes/vector'     : 256,
        '1M vectors (GB)'  : 0.256,
        'Compression'      : '12×',
        'Recall@10'        : round(recall_64, 4),
        'Quality retained' : f"{recall_64*100:.1f}%"
    },
    {
        'Version'          : '64-dim binary (true 1-bit)',
        'Bytes/vector'     : 8,
        '1M vectors (GB)'  : 0.008,
        'Compression'      : '384×',
        'Recall@10'        : round(recall_bin, 4),
        'Quality retained' : f"{recall_bin*100:.1f}%"
    },
]

df_results = pd.DataFrame(results_data)
print("\n" + "="*70)
print("FINAL RESULTS TABLE")
print("="*70)
print(df_results.to_string(index=False))
print("="*70)

# Save results table
df_results.to_csv(f"{SAVE_PATH}results_table.csv", index=False)
print(f"\n✓ Saved results_table.csv")

In [ ]:
# Check: are the "wrong" neighbors actually bad, or just equally good?
# Pick one query and inspect what baseline vs 64-dim actually retrieved

sample_idx = query_indices[0]

baseline_neighbors  = baseline_results[sample_idx]
compressed_neighbors = results_64[sample_idx]

# Load original texts to read what was retrieved
df = pd.read_parquet(f"{SAVE_PATH}electronics_meta_clean.parquet")

print(f"QUERY ({labels[sample_idx]}):")
print(f"  {df['text'].iloc[sample_idx][:150]}\n")

print(f"BASELINE TOP-5:")
for i, idx in enumerate(baseline_neighbors[:5]):
    print(f"  {i+1}. [{labels[idx]}] {df['text'].iloc[idx][:100]}")

print(f"\n64-DIM TOP-5 (overlapping with baseline marked ✓):")
baseline_set = set(baseline_neighbors)
for i, idx in enumerate(compressed_neighbors[:5]):
    mark = "✓" if idx in baseline_set else "✗"
    print(f"  {mark} {i+1}. [{labels[idx]}] {df['text'].iloc[idx][:100]}")

In [ ]:
# Find a query where binary recall = 0 (complete miss)

binary_recalls = []
for idx in query_indices:
    baseline_set    = set(baseline_results[idx])
    binary_set      = set(results_bin[idx])
    overlap         = len(baseline_set & binary_set)
    binary_recalls.append((idx, overlap))

# Sort by worst recall first
binary_recalls.sort(key=lambda x: x[1])

print("5 WORST BINARY RETRIEVALS:")
print("(queries where binary found fewest correct neighbors)\n")

for query_idx, overlap in binary_recalls[:5]:
    print(f"Query idx {query_idx} | overlap: {overlap}/10 "
          f"| category: {labels[query_idx]}")
    print(f"  Text: {df['text'].iloc[query_idx][:120]}")

    baseline_cats  = [labels[i] for i in baseline_results[query_idx][:5]]
    binary_cats    = [labels[i] for i in results_bin[query_idx][:5]]

    print(f"  Baseline top-5 categories:  {baseline_cats}")
    print(f"  Binary   top-5 categories:  {binary_cats}")
    print()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5: UMAP VISUALIZATION
#
# Goal: 3-panel UMAP showing cluster survival under compression
# GPU needed: NO
# Expected time: 10-15 minutes (UMAP is CPU-intensive)
# Expected RAM: ~3GB peak
# ═══════════════════════════════════════════════════════════════

!pip install umap-learn matplotlib -q

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import umap
import time


In [ ]:
# ── 5a. Load embeddings and labels ───────────────────────────────
embeddings    = np.load(f"{SAVE_PATH}embeddings_768_float32.npy", allow_pickle=False)
emb_64        = np.load(f"{SAVE_PATH}embeddings_064_float32.npy", allow_pickle=False)
emb_64_binary = np.load(f"{SAVE_PATH}embeddings_064_binary.npy",  allow_pickle=False)
labels        = np.load(f"{SAVE_PATH}labels.npy", allow_pickle=True)

print(f"✓ Loaded all versions")
print(f"  Baseline:  {embeddings.shape}")
print(f"  64-dim:    {emb_64.shape}")
print(f"  Binary:    {emb_64_binary.shape}")
print(f"  Labels:    {np.unique(labels)}")

In [ ]:
# ── 5b. UMAP reducer function ─────────────────────────────────────
#
# n_neighbors=30:
#   Controls how UMAP balances local vs global structure.
#   Low values (5-10) → tight local clusters, may fragment.
#   High values (50+) → more global structure, blurrier clusters.
#   30 is the sweet spot for 20K points — clusters are dense
#   and separated without being artificially tight.
#
# min_dist=0.1:
#   Controls how tightly points are packed inside clusters.
#   0.0 → maximum packing (good for clustering, bad for reading)
#   0.5 → loose packing (readable but less defined)
#   0.1 gives readable clusters with clear internal structure.
#
# n_components=2:
#   We're projecting to 2D for visualization. Always 2 for plots.
#
# random_state=42:
#   UMAP is stochastic — different seeds give different layouts.
#   Same seed across all three panels ensures the cluster
#   positions are comparable. Without this, clusters might
#   appear in different corners in each panel, making comparison
#   impossible.
#
# metric='cosine':
#   Our embeddings are L2-normalized so cosine == euclidean,
z#   For binary we use 'hamming' — the correct metric.

def fit_umap(embeddings, metric='cosine', random_state=42):
    reducer = umap.UMAP(
        n_neighbors  = 30,
        min_dist     = 0.1,
        n_components = 2,
        metric       = metric,
        random_state = random_state,
        verbose      = True
    )
    return reducer.fit_transform(embeddings)

In [ ]:
# ── 5c. Run UMAP on all three versions ───────────────────────────
# Why run separately and not reuse baseline projection?
# Each compression version has different geometry.
# Reusing the baseline projection would show the same layout
# with different colors — that's not what we want.
# We want to see how each version's OWN geometry organizes
# the data. Three independent projections, same random seed.
#
# This takes 10-15 minutes total on CPU. Normal. Don't interrupt.

print("Running UMAP on 768-dim baseline...")
t0 = time.time()
proj_768 = fit_umap(embeddings, metric='cosine')
print(f"✓ Baseline done: {(time.time()-t0)/60:.1f} min\n")

print("Running UMAP on 64-dim float32...")
t0 = time.time()
proj_64 = fit_umap(emb_64, metric='cosine')
print(f"✓ 64-dim done: {(time.time()-t0)/60:.1f} min\n")

print("Running UMAP on 64-dim binary...")
t0 = time.time()
proj_bin = fit_umap(emb_64_binary, metric='hamming')
print(f"✓ Binary done: {(time.time()-t0)/60:.1f} min\n")

print("All UMAP projections complete.")

In [ ]:
# ── 5d. Save projections immediately ──────────────────────────────
# UMAP takes 10+ minutes. Save before plotting.
# If the plotting cell crashes, we don't rerun UMAP.

np.save(f"{SAVE_PATH}umap_proj_768.npy", proj_768)
np.save(f"{SAVE_PATH}umap_proj_64.npy",  proj_64)
np.save(f"{SAVE_PATH}umap_proj_bin.npy", proj_bin)
print("✓ All projections saved to Drive")

In [ ]:
# ── 5e. Plot —  ─────────────────────────────────────

COLORS = {
    'Computers & Laptops' : '#2196F3',  # blue
    'Camera & Photo'      : '#4CAF50',  # green
    'Mobile & Phones'     : '#FF9800',  # orange
    'Audio & Home Theater': '#E91E63',  # pink
}

TITLES = [
    f"768-dim float32\n(Baseline — 3,072 bytes/vector)",
    f"64-dim float32\n(12× compression — 256 bytes/vector | Recall@10: 50.2%)",
    f"64-dim Binary\n(384× compression — 8 bytes/vector | Recall@10: 13.9%)",
]

projections = [proj_768, proj_64, proj_bin]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#0F0F0F')       # dark background

for ax, proj, title in zip(axes, projections, TITLES):
    ax.set_facecolor('#0F0F0F')

    for category, color in COLORS.items():
        mask = labels == category
        ax.scatter(
            proj[mask, 0],
            proj[mask, 1],
            c     = color,
            s     = 3,
            alpha = 0.4,
            rasterized = True    # smaller file size, smoother rendering
        )

    ax.set_title(title, color='white', fontsize=11, pad=12, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

# Legend — shared across all three panels
legend_handles = [
    mpatches.Patch(color=color, label=cat)
    for cat, color in COLORS.items()
]
fig.legend(
    handles        = legend_handles,
    loc            = 'lower center',
    ncol           = 4,
    frameon        = False,
    fontsize       = 11,
    labelcolor     = 'white',
    bbox_to_anchor = (0.5, -0.05)
)

plt.suptitle(
    "MRL + Binary Quantization: Do Semantic Clusters Survive Compression?\n"
    "20,000 Amazon Electronics Products — nomic-ai/nomic-embed-text-v1.5",
    color    = 'white',
    fontsize = 13,
    fontweight = 'bold',
    y = 1.02
)

plt.tight_layout()

plt.savefig(
    f"{SAVE_PATH}umap_comparison.png",
    dpi         = 200,
    bbox_inches = 'tight',
    facecolor   = '#0F0F0F',
    format      = 'png'
)

print("✓ Saved umap_comparison.png")
plt.show()